In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from io import StringIO
from datetime import date # para pegar a data corrente
import pyodbc 
from time import sleep


#server = '.' # Nesta instalação pode ser tanto "." como "RENATO-DESKTOP"
server = 'localhost'    # Substitua pelo nome do servidor SQL Server
database = 'Python'     # Substitua pelo nome do banco de dados
conexaoDB = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};'
                      f'SERVER={server};'
                      f'DATABASE={database};'
                      'Trusted_Connection=yes;')

cursor = conexaoDB.cursor()   # criando cursor de comando 


In [2]:
#pip install pandas
#pip install pyodbc
#pip install selenium
#pip install lxml



"""
CREATE TABLE [dbo].[fundamentus_fii](
	[Dia] [date] NULL,
	[Papel] [char](10) NULL,
	[Segmento] [nvarchar](255) NULL,
	[Cotacao] [float] NULL,
	[FFO_yield] [float] NULL,
	[Dividend_yield] [float] NULL,
	[PVP] [float] NULL,
	[Valor_mercado] [bigint] NULL,
	[Liquidez] [bigint] NULL,
	[Qtd_imoveis] [int] NULL,
	[Preco_m2] [float] NULL,
	[Aluguel_m2] [float] NULL,
	[Cap_rate] [float] NULL,
	[Vacancia_media] [float] NULL
) ON [PRIMARY]
"""

'\nCREATE TABLE [dbo].[fundamentus_fii](\n\t[Dia] [date] NULL,\n\t[Papel] [char](10) NULL,\n\t[Segmento] [nvarchar](255) NULL,\n\t[Cotacao] [float] NULL,\n\t[FFO_yield] [float] NULL,\n\t[Dividend_yield] [float] NULL,\n\t[PVP] [float] NULL,\n\t[Valor_mercado] [bigint] NULL,\n\t[Liquidez] [bigint] NULL,\n\t[Qtd_imoveis] [int] NULL,\n\t[Preco_m2] [float] NULL,\n\t[Aluguel_m2] [float] NULL,\n\t[Cap_rate] [float] NULL,\n\t[Vacancia_media] [float] NULL\n) ON [PRIMARY]\n'

In [3]:


driver = webdriver.Chrome()
driver.get("https://www.fundamentus.com.br/fii_resultado.php")

# esperar a tabela aparecer
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "tabelaResultado"))
)

# pegar o HTML da tabela
tabela = driver.find_element(By.ID, "tabelaResultado")
html_tabela = tabela.get_attribute("outerHTML")

#sleep(10)
driver.quit()



In [4]:

# converter para DataFrame
# fazer o StringIO manter centavos e milhares 
df = pd.read_html(StringIO(html_tabela), decimal=',', thousands='.')[0]

# renomeando as colunas para modo mais amigável
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.normalize('NFKD') # padrão Unicode (Unicode Normalization Form KD) para separar caracteres acentuados "ã" → "a" etc.
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

"""
Papel                object
Segmento             object
Cotação             float64
FFO Yield            object
Dividend Yield       object
P/VP                float64
Valor de Mercado      int64
Liquidez              int64
Qtd de imóveis        int64
Preço do m2         float64
Aluguel por m2      float64
Cap Rate             object
Vacância Média       object
dtype: object
"""

# converte campos percentuais (12,81%) para decimal (0,1281)
for col in df.columns:
    if df[col].astype(str).str.contains('%').any():
        df[col] = (
            df[col]
            .astype(str)
            .str.replace('%', '')
            .str.replace(',', '.')
            .replace('-', '0')
            .astype(float) / 100
        )

# inclui a data corrente, já informando que é datetime como primeira coluna do DF
df.insert(0, 'dia', pd.to_datetime( date.today() ) )


In [5]:

#print(df.dtypes)
#display(df[df['papel'].str.contains('ELDO')])
#display(df)
#df.head(20)


In [6]:
cursor.execute("delete from fundamentus_fii where dia=?", date.today() )   #executa tarefa de  apagar dados
cursor.commit()


In [7]:
for index, linha in df.iterrows():
        cursor.execute("Insert into [fundamentus_fii] (Dia, Papel, Segmento, Cotacao, FFO_yield, Dividend_yield, PVP, Valor_mercado, Liquidez, Qtd_imoveis, Preco_m2, Aluguel_m2, Cap_rate, Vacancia_media) values (?,?,?,?,?,?,?,?,?,?,?,?,?,?)", linha.dia, linha.papel, linha.segmento, linha.cotacao, linha.ffo_yield, linha.dividend_yield, linha.p_vp, linha.valor_de_mercado, linha.liquidez, linha.qtd_de_imoveis, linha.preco_do_m2, linha.aluguel_por_m2, linha.cap_rate, linha.vacancia_media)


In [8]:

cursor.commit()
cursor.close()
conexaoDB.close()